# Healthcare Dataset — Full Biostatistics & Machine Learning Analysis

This notebook provides a rigorous end-to-end analysis of a synthetic healthcare dataset,
covering data cleaning, exploratory analysis, statistical inference, survival analysis,
predictive modelling, and patient segmentation.

**Dataset:** `healthcare_dataset.csv`  
**Author:** Cyril Benson  
**Date:** June 2026  

---

### Project Structure
| Phase | Focus |
|-------|-------|
| 1 | Data Cleaning & Feature Engineering |
| 2 | Exploratory Data Analysis |
| 3 | Statistical Inference |
| 4 | Survival Analysis |
| 5 | Predictive Modelling (Multi-class Classification) |
| 6 | Patient Segmentation |

## Phase 1 — Data Cleaning & Preparation

### Step 1 — Import Libraries

We load every library needed across the full project upfront:
- `pandas` / `numpy` — data manipulation and numerical operations
- `scipy` — statistical tests (chi-square, ANOVA, Kruskal-Wallis)
- `sklearn` — machine learning models, preprocessing, and evaluation
- `lifelines` — survival analysis (Kaplan-Meier, Cox regression)
- `shap` — model explainability and feature importance
- `plotly` / `seaborn` / `matplotlib` — interactive and static visualizations
- `warnings` — suppress minor deprecation noise

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Statistical analysis
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, f_oneway

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score)
from xgboost import XGBClassifier

# Survival analysis
from lifelines import KaplanMeierFitter, CoxPHFitter

# Explainability
import shap

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# Utility
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

/opt/anaconda3/envs/healthcare312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries loaded successfully.


### Step 2 — Load the Raw Data

We read the CSV from the raw data folder and do three quick checks:
1. Shape — how many patients and columns do we have?
2. Column names and data types — are they what we expect?
3. First few rows — does the data look sensible at a glance?

In [2]:
# Load the raw dataset
df = pd.read_csv('../data/raw/healthcare_dataset.csv')

# How big is this dataset?
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Shape: 55,500 rows × 15 columns


In [3]:
# Column names and their data types
print("Column types:\n")
print(df.dtypes)

Column types:

Name                   object
Age                     int64
Gender                 object
Blood Type             object
Medical Condition      object
Date of Admission      object
Doctor                 object
Hospital               object
Insurance Provider     object
Billing Amount        float64
Room Number             int64
Admission Type         object
Discharge Date         object
Medication             object
Test Results           object
dtype: object


In [4]:
# Preview the first 5 rows
df.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


### Step 3 — Data Quality Report

Before any analysis, we need to verify the integrity of the dataset.
We check for missing values, duplicate records, incorrect data types,
and any suspicious values that could distort our findings downstream.

In [5]:
# Check for missing values across all columns
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_report)

                    Missing Count  Missing %
Name                            0        0.0
Age                             0        0.0
Gender                          0        0.0
Blood Type                      0        0.0
Medical Condition               0        0.0
Date of Admission               0        0.0
Doctor                          0        0.0
Hospital                        0        0.0
Insurance Provider              0        0.0
Billing Amount                  0        0.0
Room Number                     0        0.0
Admission Type                  0        0.0
Discharge Date                  0        0.0
Medication                      0        0.0
Test Results                    0        0.0


In [6]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

Duplicate rows: 534


In [7]:
# Unique values in categorical columns
categorical_cols = ['Gender', 'Blood Type', 'Medical Condition', 
                    'Admission Type', 'Insurance Provider', 
                    'Medication', 'Test Results']

for col in categorical_cols:
    print(f"\n{col} ({df[col].nunique()} unique):")
    print(df[col].value_counts())


Gender (2 unique):
Gender
Male      27774
Female    27726
Name: count, dtype: int64

Blood Type (8 unique):
Blood Type
A-     6969
A+     6956
AB+    6947
AB-    6945
B+     6945
B-     6944
O+     6917
O-     6877
Name: count, dtype: int64

Medical Condition (6 unique):
Medical Condition
Arthritis       9308
Diabetes        9304
Hypertension    9245
Obesity         9231
Cancer          9227
Asthma          9185
Name: count, dtype: int64

Admission Type (3 unique):
Admission Type
Elective     18655
Urgent       18576
Emergency    18269
Name: count, dtype: int64

Insurance Provider (5 unique):
Insurance Provider
Cigna               11249
Medicare            11154
UnitedHealthcare    11125
Blue Cross          11059
Aetna               10913
Name: count, dtype: int64

Medication (5 unique):
Medication
Lipitor        11140
Ibuprofen      11127
Aspirin        11094
Paracetamol    11071
Penicillin     11068
Name: count, dtype: int64

Test Results (3 unique):
Test Results
Abnormal        186

In [8]:
# Summary statistics for numeric columns
print(df[['Age', 'Billing Amount']].describe().round(2))

            Age  Billing Amount
count  55500.00        55500.00
mean      51.54        25539.32
std       19.60        14211.45
min       13.00        -2008.49
25%       35.00        13241.22
50%       52.00        25538.07
75%       68.00        37820.51
max       89.00        52764.28


#### Data Quality Summary

The dataset contains 55,500 records across 15 columns.

**Missing Values:** No missing values were found across any column.
This is expected for a synthetic dataset but is ideal for analysis.

**Duplicate Records:** 534 duplicate rows were identified (~1% of the dataset).
These will be removed during cleaning to avoid inflating patient counts
and biasing statistical results.

**Categorical Variables:** All categorical columns are clean with no
unexpected values or typos. The target variable (Test Results) is
perfectly balanced across three classes — Abnormal (33.5%), Normal (33.3%),
and Inconclusive (33.1%) — which is ideal for multi-class classification.

**Numeric Variables:** Age ranges from 13 to 89 years with a mean of 51.5,
which is clinically plausible. However, Billing Amount contains at least
one negative value (minimum: -$2,008.49), which is not clinically meaningful.
This will be investigated and resolved during data cleaning.

In [9]:
# Investigate negative billing amounts
neg_billing = df[df['Billing Amount'] < 0]
print(f"Negative billing rows: {len(neg_billing)}")
print(neg_billing['Billing Amount'].describe().round(2))

Negative billing rows: 108
count     108.00
mean     -499.30
std       426.46
min     -2008.49
25%      -792.08
50%      -369.09
75%      -153.08
max       -23.87
Name: Billing Amount, dtype: float64


#### Negative Billing Amount Investigation

108 records (0.2% of the dataset) contain negative billing amounts,
ranging from -$2,008.49 to -$23.87. The amounts themselves are plausible
— they are not outliers in magnitude — suggesting these are sign errors
rather than corrupt records.

**Decision:** Convert negative values to their absolute value using `abs()`.
This preserves all 108 records while correcting the erroneous sign.
Dropping them would also be defensible given the small number,
but imputation via absolute value is the more conservative approach.

### Step 4 — Data Cleaning & Feature Engineering

Based on the data quality report, we apply the following corrections:
1. Remove 534 duplicate rows
2. Parse date columns into proper datetime format
3. Compute Length of Stay (days between admission and discharge)
4. Fix negative billing amounts by taking the absolute value
5. Standardise column names for easier access
6. Save the cleaned dataset to data/cleaned/

In [10]:
# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)
print(f"After removing duplicates: {len(df):,} rows")

# Standardise column names — lowercase, spaces to underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(f"Columns: {list(df.columns)}")

After removing duplicates: 54,966 rows
Columns: ['name', 'age', 'gender', 'blood_type', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results']


In [11]:
# Parse date columns from string to datetime
df['date_of_admission'] = pd.to_datetime(df['date_of_admission'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])

# Compute Length of Stay in days
df['length_of_stay'] = (df['discharge_date'] - df['date_of_admission']).dt.days

print(f"Length of Stay — min: {df['length_of_stay'].min()} days, "
      f"max: {df['length_of_stay'].max()} days, "
      f"mean: {df['length_of_stay'].mean():.1f} days")

Length of Stay — min: 1 days, max: 30 days, mean: 15.5 days


In [12]:
# Convert negative billing amounts to absolute value
df['billing_amount'] = df['billing_amount'].abs()

print(f"Billing Amount — min: ${df['billing_amount'].min():,.2f}, "
      f"max: ${df['billing_amount'].max():,.2f}, "
      f"mean: ${df['billing_amount'].mean():,.2f}")

Billing Amount — min: $9.24, max: $52,764.28, mean: $25,546.24


In [13]:
# Save the cleaned dataset
df.to_csv('../data/cleaned/healthcare_cleaned.csv', index=False)
print(f"Cleaned dataset saved — {len(df):,} rows × {df.shape[1]} columns")

Cleaned dataset saved — 54,966 rows × 16 columns
